# Пространственные отношения


В предыдущем разделе мы научились изменять геометрию объектов и создавать новые пространственные данные. Однако во многих задачах важно не только преобразовывать геометрию, но и анализировать, **как объекты расположены относительно друг друга**.

Пространственные отношения позволяют определить, пересекаются ли объекты, находятся ли они внутри других объектов или касаются ли они друг друга. Эти операции лежат в основе многих задач пространственного анализа, таких как:

- поиск объектов в заданной зоне;
- определение доступности;
- фильтрация данных по пространственному признаку.

В GeoPandas для этого используются специальные методы, называемые **пространственными предикатами** (spatial predicates), например: `intersects`, `within`, `contains`.

В этом разделе мы рассмотрим, как применять пространственные отношения для анализа и фильтрации геоданных.


## 0. Импортируем библиотеки и подготавливаем данные


### 0.1 Импорт библиотек


In [2]:
import pandas as pd  
import geopandas as gpd
import osmnx as ox

- [**pandas**](https://pandas.pydata.org/) (`pandas`) — библиотека Python для работы с табличными данными.

- [**GeoPandas**](https://geopandas.org/) (`geopandas`) — библиотека Python, расширение библиотеки pandas, предназначенное для работы с геопространственными данными. Позволяет загружать, обрабатывать и анализировать пространственные наборы данных в различных форматах.

- [**OSMnx**](https://osmnx.readthedocs.io/) (`osmnx`) — библиотека Python для загрузки, анализа и визуализации данных **OpenStreetMap**.
  Она позволяет легко получать пространственные объекты по названию места или по координатам.


### 0.2 Подготовка данных


Загрузим границу Центрального района Санкт-Петербурга из OpenStreetMap


In [ ]:
area_name = "Центральный район, Санкт-Петербург"
admin_border = ox.geocode_to_gdf(area_name)

admin_border.explore(tiles="cartodbpositron")

Прочитаем данные о театрах Санкт-Петербурга из CSV-файла и создадим на их основе `GeoDataFrame`.


In [ ]:
theaters_csv = pd.read_csv('./data/spb_theaters.csv')

theaters_gpd = gpd.GeoDataFrame(
    theaters_csv,
    geometry=gpd.points_from_xy(theaters_csv['longitude'], theaters_csv['latitude']),
    crs = 4326
)

## 1. Пространственные предикаты

**Пространственные предикаты** – это функции, позволяющие определить топологическое отношение между двумя геометрическими объектами (точками, линиями, полигонами и т.д.). Предикаты позволяют отвечать на вопросы вроде: «Пересекаются ли эти объекты?», «Находится ли один объект внутри другого?», «Касаются ли они границ друг друга?» и т.п. Знание пространственных отношений помогает выполнять пространственные запросы и объединения данных (spatial join), фильтрацию на основе взаимного расположения и др.

### 1.1 Описание пространственных предикатов

Ниже приведена таблица основных бинарных предикатов (отношений) с кратким описанием:

| **Предикат**                     | **Описание** (условие истинности)                                                                                                                                                                                                                                                                 |
| -------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **intersects** (_пересекается_)  | **Имеют общую точку** – Геометрии пересекаются, если они имеют хотя бы одну общую точку в пространстве (любое наложение границ или внутренних областей). Эквивалентно логическому отрицанию disjoint (см. ниже).                                                                                  |
| **disjoint** (_не пересекается_) | **Не имеют общих точек** – Геометрии не пересекаются, если у них **нет ни одной** общей точки. То есть полностью разделены в пространстве.                                                                                                                                                        |
| **within** (_внутри_)            | **A внутри B** – Геометрия A находится целиком внутри геометрии B, не выходя за ее границы. Все точки A (и граница, и внутренняя часть) лежат во внутренней области B. (Эквивалентно B.contains(A)).                                                                                              |
| **contains** (_содержит_)        | **A содержит B** – Геометрия A полностью содержит геометрию B. То есть ни одна часть B не лежит вне A, при этом B не просто граничит с A, а имеет хотя бы одну точку строго внутри A. (Эквивалентно B.within(A)).                                                                                 |
| **touches** (_касается_)         | **Касание границ** – Геометрии имеют хотя бы одну общую точку на границе, но **не** пересекаются по внутренней области. Иными словами, их границы соприкасаются, но пересечения интерьеров нет.                                                                                                   |
| **overlaps** (_перекрывает_)     | **Частичное перекрытие** – Геометрии частично перекрываются, если они имеют общую часть, но ни одна не содержит другую целиком. При этом геометрии одного типа (полигон-полигон или линия-линия) – например, два полигона, которые пересекаются по площади, но каждый выходит за пределы другого. |
| **crosses** (_перекрещивается_)  | **Пересечение с выходом** – Геометрии пересекаются _внутренними точками_ таким образом, что пересечение имеет меньшую размерность. Например, линия пересекает полигон, входя и выходя из него (часть линии внутри, часть снаружи), или две линии пересекаются в одной точке.                      |


### 1.2 Пространственные предикаты в GeoPandas


В GeoPandas пространственные предикаты реализованы в виде методов, которые применяются к геометриям и возвращают логические значения (`True` или `False`).

Это означает, что для каждой геометрии проверяется выполнение условия пространственного отношения.

Проверим, какие театры находятся внутри Центрального района Санкт-Петербурга


In [17]:
theaters_gpd.within(admin_border.geometry.iloc[0])

0       True
1       True
2       True
3       True
4       True
       ...  
104     True
105     True
106     True
107     True
108    False
Length: 109, dtype: bool

Результатом будет серия значений `True/False`, где:

- `True` — объект удовлетворяет условию (находится внутри);
- `False` — не удовлетворяет.

Рассмотрим, как пространственные предикаты применяются на практике для фильтрации геоданных.


## 2. Фильтрация объектов по пространственному признаку


Одно из распространённых действий в пространственном анализе – отбор объектов, удовлетворяющих определённому пространственному условию. Например, можно захотеть **выбрать все точки (объекты) внутри заданного полигона**. Это называется пространственной фильтрацией (фильтрация по местоположению).

Давайте отфильтруем все театры, которые находятся в Центральном районе Санкт-Петербурга


In [15]:
theaters_center = theaters_gpd[
    theaters_gpd.geometry.within(admin_border.geometry.iloc[0])
]

In [16]:
theaters_center.explore(tiles='CartoDB positron')

На карте видно, что отобраны только те театры, которые находятся в Центральном районе Санкт-Петербурга.


## Итог

В этом разделе мы познакомились с пространственными отношениями и научились применять их для анализа геоданных.

Мы узнали:

- что такое пространственные предикаты и какие типы отношений существуют;
- как использовать предикаты в GeoPandas;
- как фильтровать объекты на основе их пространственного расположения.

В рассмотренных примерах мы использовали пространственные предикаты для фильтрации данных. Однако при работе с несколькими наборами данных часто требуется не только отобрать объекты, но и объединить их атрибутивную информацию. Для этого используется операция пространственного соединения (spatial join), которую мы рассмотрим в следующем разделе.
